# PyTorch on WebAssembly (Pyodide) — train a small MLP in the browser

This notebook runs entirely in your browser via a **Pyodide** kernel (no server).
It installs a from-source **CPU-only `torch` wheel built for `wasm32-emscripten`**
and trains a small multi-layer perceptron with **autograd + SGD**.

The wheel is a reduced build (single-threaded, no XNNPACK/MKLDNN/distributed/CUDA),
but eager autograd, `torch.nn`, `torch.optim` are compiled in. See the repo
[`RESULTS.md`](../../RESULTS.md) for the full build story.

## Status (honest)

The bootstrap cell below loads the wheel's wasm side modules **in dependency order**.
As of this build:

* ✅ **Blocker #10 solved** — `libc10.so` loads and exports its C++ symbols
  (`c10::getRuntimeDispatchKeySet`), so cross-`.so` symbol resolution succeeds.
* ⚠️ **Blocker #11 remaining** — loading the 82 MB `libtorch_cpu.so` aborts inside a
  C++ static initializer (Emscripten `invoke_viii` → `getWasmTableEntry(...) is not a
  function`: an unresolved `GOT.func` pointer). The bootstrap reports this clearly.

The training cells (2–5) are the intended demonstration and run end-to-end once
blocker #11 is resolved.

In [ ]:
# Bootstrap: install the torch wheel and load its wasm side modules in dependency
# order (micropip's auto-loader looks in the wrong dir, so we load explicitly).
import sys, js, pyodide_js, piplite

SP = f"/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages"

# 1) Install torch's pure-Python deps + unpack the torch wheel's Python tree.
#    (piplite.install may raise while auto-loading the .so from the wrong dir; we
#    still get the extracted package tree, then load the libraries ourselves.)
try:
    await piplite.install(["typing-extensions", "sympy", "mpmath", "networkx", "jinja2", "fsspec"])
except Exception as e:
    print("dep install warning:", e)
try:
    await piplite.install("torch", deps=False)
except Exception as e:
    print("(expected) torch auto-load raised; continuing with manual load:", str(e)[:120])

# 2) Load the side modules in dependency order.
LIBS = [
    "torch/lib/libc10.so",
    "torch/lib/libtorch_cpu.so",
    "torch/lib/libtorch.so",
    "torch/lib/libshm.so",
    "torch/lib/libtorch_python.so",
    "torch/lib/libtorch_global_deps.so",
    "torch/_C.cpython-312-wasm32-emscripten.so",
    "functorch/_C.cpython-312-wasm32-emscripten.so",
]
SEARCH = [f"{SP}/torch/lib", f"{SP}/torch", f"{SP}/functorch"]
loaded_ok = True
for rel in LIBS:
    try:
        await pyodide_js._api.loadDynlib(f"{SP}/{rel}", True, SEARCH)
        print("loaded", rel)
    except Exception as e:
        loaded_ok = False
        print("\nBLOCKED loading", rel)
        print("  ->", str(e).splitlines()[0][:200] if str(e) else repr(e))
        print("  This is blocker #11 (unresolved GOT.func pointer in a C++ static")
        print("  initializer). Blocker #10 (symbol export) is solved: libc10 loaded above.")
        break

if loaded_ok:
    import torch
    print("\ntorch", torch.__version__, "| default dtype", torch.get_default_dtype())
else:
    print("\ntorch import not attempted (see blocker above). Training cells will run")
    print("once the remaining side module loads.")

In [ ]:
# A tiny synthetic 2D binary-classification dataset (two Gaussian blobs).
import torch
torch.manual_seed(0)
N = 256
c0 = torch.randn(N, 2) * 0.6 + torch.tensor([-1.5, -1.5])
c1 = torch.randn(N, 2) * 0.6 + torch.tensor([ 1.5,  1.5])
X = torch.cat([c0, c1], dim=0)
y = torch.cat([torch.zeros(N), torch.ones(N)]).long()
print("X", tuple(X.shape), "y", tuple(y.shape))

In [ ]:
# Define a small MLP: Linear -> ReLU -> Linear.
import torch.nn as nn
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
print(model)

In [ ]:
# Train for a few epochs; record the loss curve.
losses = []
for epoch in range(60):
    opt.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()      # autograd
    opt.step()           # SGD update
    losses.append(float(loss))
    if epoch % 10 == 0 or epoch == 59:
        acc = (logits.argmax(1) == y).float().mean().item()
        print(f"epoch {epoch:3d}  loss={float(loss):.4f}  acc={acc:.3f}")
print("final loss", losses[-1])
assert losses[-1] < losses[0], "loss should decrease"

In [ ]:
# Plot the loss curve (matplotlib ships with Pyodide).
import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.plot(losses)
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss")
plt.title("Training an MLP with PyTorch (wasm) in the browser")
plt.tight_layout(); plt.show()